# ACPE Match — Seed Database

Ce notebook lance le seed complet de la base ACPE Match :
- 41,285 candidats + ~2,500 offres
- Embeddings BGE-M3 (GPU)
- SQLite + ChromaDB + FAISS

**Durée estimée** : ~1-2h avec GPU P100/T4

## Étape 1 — Installation des dépendances

In [ ]:
!pip install -q sentence-transformers tf-keras chromadb rapidfuzz faiss-cpu catboost openpyxl sqlalchemy

import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"RAM GPU : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Étape 2 — Cloner le projet

In [ ]:
!git clone https://github.com/Prosper-BATAMBA/ACPE-Match.git /kaggle/working/ACPE-Match
%cd /kaggle/working/ACPE-Match
!git log --oneline -3

## Étape 3 — Upload des données Excel

Upload `Demandeurs.xlsx` et `Offres_ACPE.xlsx` dans `/kaggle/input/` via Kaggle UI :
- Clique sur **+ Add Data** en haut à droite
- Sélectionne **Upload** 
- Upload les 2 fichiers Excel

Puis modifie le chemin ci-dessous si nécessaire.

In [ ]:
import os

# Chemins source (upload via Kaggle UI)
KAGGLE_INPUT = "/kaggle/input"
DEMANDEURS_SRC = None
OFFRES_SRC = None

# Cherche les fichiers uploadés
for root, dirs, files in os.walk(KAGGLE_INPUT):
    for f in files:
        if "Demandeur" in f and f.endswith(".xlsx"):
            DEMANDEURS_SRC = os.path.join(root, f)
        if "Offre" in f and f.endswith(".xlsx"):
            OFFRES_SRC = os.path.join(root, f)

if not DEMANDEURS_SRC:
    print("ERREUR: Demandeurs.xlsx non trouvé. Upload-le via + Add Data > Upload")
else:
    print(f"Demandeurs: {DEMANDEURS_SRC}")

if not OFFRES_SRC:
    print("ERREUR: Offres_ACPE.xlsx non trouvé. Upload-le via + Add Data > Upload")
else:
    print(f"Offres: {OFFRES_SRC}")

In [ ]:
# Copie les fichiers vers le bon répertoire du projet
RAW_DIR = "/kaggle/working/ACPE-Match/matching_engine/matching_engine/data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

if DEMANDEURS_SRC:
    !cp "{DEMANDEURS_SRC}" "{RAW_DIR}/Demandeurs.xlsx"
    print(f"Copié: {RAW_DIR}/Demandeurs.xlsx")

if OFFRES_SRC:
    !cp "{OFFRES_SRC}" "{RAW_DIR}/Offres_ACPE.xlsx"
    print(f"Copié: {RAW_DIR}/Offres_ACPE.xlsx")

# Vérifie
!ls -lh "{RAW_DIR}/"

## Étape 4 — Lancer le seed

Cela va :
1. Créer SQLite `acpe.db` (candidats + offres)
2. Encoder tous les profils avec BGE-M3 (GPU)
3. Stocker les vecteurs dans ChromaDB

**Durée** : ~1-2h avec GPU

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"

%cd /kaggle/working/ACPE-Match/backend
!python -m seed_data

## Étape 5 — Construire l'index FAISS

In [ ]:
%cd /kaggle/working/ACPE-Match/backend
!python build_faiss_index.py

## Étape 6 — Vérification rapide

In [ ]:
import sqlite3
import chromadb
import json
import os

BACKEND = "/kaggle/working/ACPE-Match/backend"

# SQLite
conn = sqlite3.connect(os.path.join(BACKEND, "acpe.db"))
n_cand = conn.execute("SELECT COUNT(*) FROM candidates").fetchone()[0]
n_off = conn.execute("SELECT COUNT(*) FROM job_offers").fetchone()[0]
conn.close()
print(f"SQLite: {n_cand} candidats, {n_off} offres")

# ChromaDB
chroma = chromadb.PersistentClient(path=os.path.join(BACKEND, "chroma_data"))
n_cand_emb = chroma.get_collection("candidate_embeddings").count()
n_off_emb = chroma.get_collection("offer_embeddings").count()
print(f"ChromaDB: {n_cand_emb} candidats, {n_off_emb} offres embeddés")

# FAISS
meta_path = os.path.join(BACKEND, "faiss_offers_meta.json")
if os.path.exists(meta_path):
    meta = json.load(open(meta_path))
    print(f"FAISS: {meta['n_offers']} offres indexées, dim={meta['dim']}")
else:
    print("FAISS: index non trouvé")

## Étape 7 — Package et téléchargement

Zip tous les fichiers nécessaires pour les réimporter dans ton projet local.

In [ ]:
import shutil
import os

BACKEND = "/kaggle/working/ACPE-Match/backend"
OUTPUT = "/kaggle/working"

# Fichiers à zipper
files_to_zip = [
    (os.path.join(BACKEND, "acpe.db"), "acpe.db"),
    (os.path.join(BACKEND, "faiss_offers.index"), "faiss_offers.index"),
    (os.path.join(BACKEND, "faiss_offers_meta.json"), "faiss_offers_meta.json"),
]

dirs_to_zip = [
    (os.path.join(BACKEND, "chroma_data"), "chroma_data"),
]

# Copie tout dans un dossier propre
DIST = os.path.join(OUTPUT, "acpe_seed_output")
os.makedirs(DIST, exist_ok=True)

for src, name in files_to_zip:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DIST, name))
        print(f"Copié: {name}")
    else:
        print(f"Manquant: {name}")

for src, name in dirs_to_zip:
    if os.path.exists(src):
        dst = os.path.join(DIST, name)
        shutil.copytree(src, dst)
        print(f"Copié: {name}/")
    else:
        print(f"Manquant: {name}/")

# Zip
ZIP_PATH = os.path.join(OUTPUT, "acpe_seed_output")
shutil.make_archive(ZIP_PATH, 'zip', DIST)

zip_size = os.path.getsize(f"{ZIP_PATH}.zip") / 1e6
print(f"\nZIP créé: {ZIP_PATH}.zip ({zip_size:.0f} MB)")
print(f"\nTélécharge-le via le bouton Kaggle ( Files > acpe_seed_output.zip )")

## Étape 8 — (Optionnel) Test rapide du matching

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/ACPE-Match/backend")
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

from matching_engine import SkillNormalizer, SpecialtyNormalizer, JobNormalizer, SectorNormalizer

sn = SkillNormalizer()
jn = JobNormalizer()
secn = SectorNormalizer()

# Test extraction compétences
test_cv = "Je suis comptable SYSCOHADA, je maîtrise Excel et les déclarations fiscales"
skills = sn.extract_from_text(test_cv)
print("Compétences détectées:")
for s in skills:
    print(f"  - {s['libelle_canonique']} ({s['type']})")

# Test normalisation métier
result = jn.normalize("Comptable")
print(f"\nMétier 'Comptable' -> {result.get('id_famille')}")

result = secn.normalize("Transport, Logistique")
print(f"Secteur 'Transport, Logistique' -> {result.get('id_secteur')}")